# DPS Step Memory Profiler
Measures peak GPU memory of one full gradient computation step.
Imports directly from `run_dps.py` and `generation.py`.

## 1. Config

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────
NUM_VARIATIONS = 6        # number of sprinter variations per step
N_TARGETS      = 6        # number of target CLIP embeddings
BASE_ZETA      = 1.0

ARCHITECT_MODEL_ID  = 'stabilityai/sdxl-turbo'
SPRINTER_MODEL_ID   = 'stabilityai/sdxl-turbo'
CONTROLNET_MODEL_ID = 'xinsir/controlnet-scribble-sdxl-1.0'
LORA_PATH           = None

VARIATION_PROMPT     = 'a superrealistic professional photograph of'
TARGET_MAN_PROMPT   = 'a superrealistic portrait photograph of a man, studio lighting'
TARGET_WOMAN_PROMPT = 'a superrealistic portrait photograph of a woman, studio lighting'
# ──────────────────────────────────────────────────────────────

import sys
sys.path.insert(0, 'SD_cond_SD_controlnet')

## 2. Memory helpers

In [ ]:
import gc
import torch

def reset_mem():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def current_mb():
    return torch.cuda.memory_allocated() / 1e6

def peak_mb():
    return torch.cuda.max_memory_allocated() / 1e6

def report(label, baseline_mb):
    cur  = current_mb()
    peak = peak_mb()
    print(f'  {label}')
    print(f'    current : {cur:.0f} MB')
    print(f'    peak    : {peak:.0f} MB')
    print(f'    net peak: {peak - baseline_mb:.0f} MB  (above pre-step baseline)')
    return peak

## 3. Load models

In [ ]:
from models import load_models, setup_gradient_checkpointing
from clip_utils import load_clip_model, encode_images_clip

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
print(f'GPU    : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

reset_mem()
architect, sprinter = load_models(
    device,
    architect_lora_path=LORA_PATH,
    architect_model_id=ARCHITECT_MODEL_ID,
    sprinter_model_id=SPRINTER_MODEL_ID,
    controlnet_model_id=CONTROLNET_MODEL_ID,
)
clip_model, clip_processor = load_clip_model(device)
sprinter.vae.to(dtype=torch.float32)
setup_gradient_checkpointing(architect, sprinter)

MODELS_MB = current_mb()
print(f'\nModels loaded. VRAM used: {MODELS_MB:.0f} MB')

## 4. Build targets + scribble

In [ ]:
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import numpy as np

from image_utils import build_base_image, sobel_proxy
from generation import generate_and_store_cs
from run_dps   import extract_scribble_hed, pil_images_to_tensor

# Build oval scribble as controlnet conditioning
_, base_tensor = build_base_image(device)
with torch.no_grad():
    sobel_cond_pil = T.ToPILImage()(sobel_proxy(base_tensor, device).squeeze(0).cpu())

# Generate target images and encode to CLIP
n_half = N_TARGETS // 2
print(f'Generating {N_TARGETS} target images...')
with torch.no_grad():
    man_imgs, _   = generate_and_store_cs(sprinter, TARGET_MAN_PROMPT,   sobel_cond_pil, n_half, batch_size=2, cn_scale=0.5)
    woman_imgs, _ = generate_and_store_cs(sprinter, TARGET_WOMAN_PROMPT, sobel_cond_pil, n_half, batch_size=2, cn_scale=0.5)

    man_embs   = encode_images_clip(pil_images_to_tensor(man_imgs,   device), clip_model, clip_processor)
    woman_embs = encode_images_clip(pil_images_to_tensor(woman_imgs, device), clip_model, clip_processor)

all_clip_embeddings = torch.cat([man_embs, woman_embs], dim=0)  # [N_TARGETS, 768]
print(f'Target CLIP embeddings: {all_clip_embeddings.shape}')

# Extract HED scribble from first target
scribble_pil = extract_scribble_hed(man_imgs[0])
print('Scribble ready.')

## 5. Prepare scheduler + latents for step 0

In [ ]:
import copy
from generation import predict_noise_cfg, compute_pred_x0_direct

height, width = 512, 512
n_steps    = 30
start_step = 15

with torch.no_grad():
    prompt_embeds, neg_embeds, pooled_embeds, neg_pooled = architect.encode_prompt(
        prompt='', negative_prompt='', device=device,
        do_classifier_free_guidance=True, num_images_per_prompt=1,
    )

architect.scheduler.set_timesteps(n_steps, device=device)
timesteps = architect.scheduler.timesteps

add_time_ids = torch.tensor(
    [[height, width, 0, 0, height, width]], dtype=prompt_embeds.dtype, device=device
)
added_cond_kwargs = {
    'text_embeds': torch.cat([neg_pooled, pooled_embeds], dim=0),
    'time_ids':    add_time_ids.repeat(2, 1),
}
cfg_encoder_states = torch.cat([neg_embeds, prompt_embeds], dim=0)

# SDEdit-style init: VAE encode scribble -> noise to start_step
with torch.no_grad():
    scribble_t  = TF.to_tensor(scribble_pil).unsqueeze(0).to(device).float()
    scribble_t  = scribble_t * 2.0 - 1.0
    scribble_lat = architect.vae.encode(scribble_t).latent_dist.mean
    scribble_lat = scribble_lat * architect.vae.config.scaling_factor

t_start = timesteps[start_step]
alpha   = architect.scheduler.alphas_cumprod[t_start.long()].to(device).float()
noise   = torch.randn_like(scribble_lat)
latents = ((alpha**0.5) * scribble_lat + ((1-alpha)**0.5) * noise).to(torch.float16)

# Use the first timestep to run
t = timesteps[start_step]
print(f'Latents shape : {latents.shape}')
print(f'Timestep      : t={t.item():.0f}')
print(f'VRAM before step: {current_mb():.0f} MB')

## 6. Measure gradient computation memory
This cell runs the full DPS gradient step and reports memory at each sub-stage.

In [ ]:
from generation import run_dps_step_clip
from metrics    import compute_mmd

# ── Snapshot baseline (models loaded, no grad graph yet) ──────────────
reset_mem()
BASELINE_MB = current_mb()
print(f'Baseline VRAM (models only): {BASELINE_MB:.0f} MB')
print()

# ── Stage 1: architect UNet forward (builds grad graph on latents_step) ──
latents_step = latents.detach().requires_grad_(True)

noise_pred = predict_noise_cfg(
    architect.unet, architect.scheduler,
    latents_step, t, cfg_encoder_states, added_cond_kwargs, guidance_scale=0.0,
)
report('After architect UNet forward (noise_pred)', BASELINE_MB)
print()

# ── Stage 2: pred_x0 formula ──────────────────────────────────────────
pred_x0 = compute_pred_x0_direct(architect.scheduler, noise_pred, t, latents_step)
report('After compute_pred_x0_direct', BASELINE_MB)
print()

# ── Stage 3: VAE decode pred_x0 -> pixels ─────────────────────────────
pred_x0_scaled = pred_x0 / architect.vae.config.scaling_factor
pixel_x0 = torch.utils.checkpoint.checkpoint(
    lambda lat: architect.vae.decode(lat.to(architect.vae.dtype)).sample,
    pred_x0_scaled, use_reentrant=False,
)
pixel_x0_norm = torch.clamp((pixel_x0.float() + 1.0) / 2.0, 0.0, 1.0)
report('After VAE decode -> pixel_x0_norm', BASELINE_MB)
print()

# ── Stage 4: full run_dps_step_clip  ──────────────────────────────────
# This is the expensive part: N x (sprinter 2-step + VAE + CLIP) + MMD + autograd.grad
print(f'Running run_dps_step_clip with num_variations={NUM_VARIATIONS}...')
grad, mmd_loss, zeta_i, loss_norm, vl_clip_flat = run_dps_step_clip(
    latents           = latents,
    latents_step      = latents_step,
    noise_pred        = noise_pred,
    pixel_x0_norm     = pixel_x0_norm,
    sprinter          = sprinter,
    all_clip_embeddings   = all_clip_embeddings,
    num_variations        = NUM_VARIATIONS,
    variation_batch_size  = 1,
    base_zeta_prime       = BASE_ZETA,
    clip_model            = clip_model,
    clip_processor        = clip_processor,
    vae                   = sprinter.vae,
    vae_scaling_factor    = sprinter.vae.config.scaling_factor,
    variation_prompt      = VARIATION_PROMPT,
    loss_fn               = compute_mmd,
)
print()
report('After full run_dps_step_clip (incl. autograd.grad)', BASELINE_MB)

print()
print('─'*50)
print(f'  grad norm  : {grad.norm().item():.6f}')
print(f'  MMD loss   : {mmd_loss.item():.6f}')
print(f'  num_vars   : {NUM_VARIATIONS}')
print(f'  n_targets  : {N_TARGETS}')
print(f'  sprinter steps per variation: 2  (SDXL Turbo)')
print(f'  equivalent standard diffusion (30 steps) would be ~15x the variation memory')
print('─'*50)

## 7. Summary

In [ ]:
peak = peak_mb()
net  = peak - BASELINE_MB
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e6

print('=== Memory Summary ===')
print(f'Total GPU VRAM         : {total_vram:.0f} MB')
print(f'Models baseline        : {BASELINE_MB:.0f} MB')
print(f'Peak during DPS step   : {peak:.0f} MB')
print(f'Net cost of grad step  : {net:.0f} MB')
print(f'Headroom remaining     : {total_vram - peak:.0f} MB')
print()
print('=== Projection: standard diffusion (30 steps) instead of Turbo (2 steps) ===')
# Net cost of the variation forward passes scales with num_inference_steps.
# Everything else (architect UNet, VAE decode of pred_x0, MMD) stays fixed.
# We isolate the variation cost by subtracting the fixed part measured at stage 3.
fixed_mb      = BASELINE_MB   # conservative: just models
variation_net = net           # total net cost (dominated by variations)
projected_30  = BASELINE_MB + variation_net * (30 / 2)
print(f'Projected peak (30 steps): {projected_30:.0f} MB')
print(f'Fits in GPU?             : {"YES" if projected_30 < total_vram else "NO — would OOM"}')